In [7]:
import sys
import xarray as xr
import numpy as np
from matplotlib import pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature
import geopandas as gpd
from shapely.geometry import mapping
import pandas as pd
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [8]:
datap = "/Users/ellendyer/Documents/GitHub/Isotopes_F4R/plots/"
datael = "/Volumes/ESA_F4R/era/era5/pressure_levels/"
dataes = "/Volumes/ESA_F4R/era/era5/surface/"
dataep = "/Volumes/ESA_F4R/era/era5/era5_surface/"
band = {'N':[5,12,10,31],'EQ':[-5,5,8,29],'S':[-15,-5,12,31]}
Y1=1990
Y2=2024
B = 'EQ'
lon_bnds, lat_bnds = (band[B][2], band[B][3]), (band[B][1],band[B][0])
time_bnds = (str(Y1)+'-01-01',str(Y2)+'-12-31')

from functools import partial
def _preprocess_pres_avg(x, lon_bnds, lat_bnds):
    return x.sel(lon=slice(*lon_bnds), lat=slice(*lat_bnds),drop=True).resample(time='D').mean(dim=('time','lat','lon')).load()
partial_func_pres_avg = partial(_preprocess_pres_avg, lon_bnds=lon_bnds, lat_bnds=lat_bnds)
    
from functools import partial
def _preprocess_pres(x, lon_bnds, lat_bnds):
    return x.sel(lon=slice(*lon_bnds), lat=slice(*lat_bnds),drop=True).resample(time='D').mean(dim=('time')).load()
partial_func_pres = partial(_preprocess_pres, lon_bnds=lon_bnds, lat_bnds=lat_bnds)
    
def _preprocess_surf(x, lon_bnds, lat_bnds):
        x=x.rename({'valid_time':'time','latitude':'lat','longitude':'lon'})
        x = x.sel(lon=slice(*lon_bnds), lat=slice(*lat_bnds),drop=True).resample(time='D').mean(dim=('time')).load()
        return x
partial_func_surf = partial(_preprocess_surf, lon_bnds=lon_bnds, lat_bnds=lat_bnds)

In [9]:
ds_era_surf_avg = xr.open_mfdataset(dataes+"era5_surface_variables_central_africa_*.nc",
                                    preprocess=partial_func_surf,parallel=True).mean(dim=('lat','lon'))
ds_era_surf_avg.to_netcdf("/Volumes/ESA_F4R/ed_prepare/analysis/ds_era_surf_avg.nc", mode='w', format='NETCDF4', engine='h5netcdf')

In [10]:
ds_era_psfc = xr.open_mfdataset(dataep+"era5_surface_pressure_central_africa_*.nc",
                                    preprocess=partial_func_surf,parallel=True)['sp']
sp = ds_era_psfc/100.0
sp = sp.sortby('lat', ascending=True)
sp.to_netcdf("/Volumes/ESA_F4R/ed_prepare/analysis/ds_era_psfc.nc", mode='w', format='NETCDF4', engine='h5netcdf')

In [12]:
#Reading in pressure level variables from ERA5 - area average
for Y in range(Y1,Y2+1):
    print(Y)
    ds_era_pres_avg = xr.open_mfdataset(datael+"era5_pressure_level_variables_central_africa_*"+str(Y)+"*.nc",
                                    drop_variables=['w','u','v'],chunks={'time':24},
                                    preprocess=partial_func_pres_avg,parallel=True)
    ds_era_pres_avg.to_netcdf("/Volumes/ESA_F4R/ed_prepare/analysis/ds_era_pres_avg_"+str(Y)+".nc", mode='w', format='NETCDF4', engine="h5netcdf")

1990
1991
1992
1993
1994
1995
1996
1997
1998
1999
2000
2001
2002
2003
2004
2005
2006
2007
2008
2009
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023
2024


In [ ]:
#Reading in pressure level variables from ERA5 - no area average
ds_era_pres = xr.open_mfdataset(datael+"era5_pressure_level_variables_central_africa_\\*.nc",
                                    preprocess=partial_func_pres,parallel=True)
ds_era_pres = ds_era_pres.sortby('lat', ascending=True)
ds_era_pres.to_netcdf("/Volumes/ESA_F4R/ed_prepare/analysis/ds_era_pres.nc", mode='w', format='NETCDF4', engine='h5netcdf')